In [ ]:
%pip install                                                         \
        numpy                 python-dotenv       openai             \
        adlfs                 fsspec              azure-storage-blob \
        azure-ai-inference

%restart_python

In [ ]:
import os
import json
import asyncio
import httpx
import numpy  as np
import scipy  as sp
from dotenv   import load_dotenv
from logging  import Logger, getLogger
from openai   import OpenAI, AsyncOpenAI

from azure.ai.inference.models import SystemMessage, UserMessage
from sentence_transformers     import SentenceTransformer


from llm_agent   import LlmAgent

In [ ]:
# 手入力
BRAND_NAME      = ''
BRAND_NAME_FILE = ''


In [ ]:
# .env ファイルを読み込む
load_dotenv()

# 環境変数の取得
STORAGE_CONNECTION_STRING = os.environ.get("STORAGE_CONNECTION_STRING")
STORAGE_CONTAINER_NAME    = os.environ.get("STORAGE_CONTAINER_NAME")
AI_FOUNDRY_ENDPOINT       = os.environ.get("AI_FOUNDRY_ENDPOINT")
AI_FOUNDRY_API_KEY        = os.environ.get("AI_FOUNDRY_API_KEY")
AI_FOUNDRY_MODEL          = os.environ.get("AI_FOUNDRY_MODEL")
LLM_MAX_TOKENS            = os.environ.get("LLM_MAX_TOKENS")
LLM_TEMPERATURE           = os.environ.get("LLM_TEMPERATURE")
LLM_TOP_P                 = os.environ.get("LLM_TOP_P")


# メモ：
# 環境変数・ウィジット経由でのパラメータの取得方法では、無条件にパラメータの型がStringに変換されてしまう
# そのため、受け取ったパラメータを適切に型変換する必要がある
LLM_MAX_TOKENS       = int(LLM_MAX_TOKENS)
LLM_TEMPERATURE      = float(LLM_TEMPERATURE)
LLM_TOP_P            = float(LLM_TOP_P)

# 簡易デバッグ用
# print(f'STORAGE_CONNECTION_STRING: {STORAGE_CONNECTION_STRING}') # セキュリティリスクのため、コメントアウト
# print(f'STORAGE_CONTAINER_NAME:    {STORAGE_CONTAINER_NAME}')    # セキュリティリスクのため、コメントアウト
# print(f'AI_FOUNDRY_ENDPOINT:       {AI_FOUNDRY_ENDPOINT}')       # セキュリティリスクのため、コメントアウト
# print(f'AI_FOUNDRY_API_KEY:        {AI_FOUNDRY_API_KEY}')        # セキュリティリスクのため、コメントアウト
print(f'AI_FOUNDRY_MODEL:          {AI_FOUNDRY_MODEL}')
print(f'LLM_MAX_TOKENS:            {LLM_MAX_TOKENS}')
print(f'LLM_TEMPERATURE:           {LLM_TEMPERATURE}')
print(f'LLM_TOP_P:                 {LLM_TOP_P}')

In [ ]:
limits      = httpx.Limits(max_keepalive_connections=20, max_connections=300)
timeout     = httpx.Timeout(300.0, connect=5.0)
http_client = httpx.AsyncClient(limits=limits, timeout=timeout)
openai_cli  = AsyncOpenAI(
                    base_url=AI_FOUNDRY_ENDPOINT,
                    api_key=AI_FOUNDRY_API_KEY,
                    http_client=http_client,
                    max_retries=3
                )
llmClient   = LlmAgent(openai_cli, AI_FOUNDRY_MODEL, int(LLM_MAX_TOKENS), float(LLM_TEMPERATURE), float(LLM_TOP_P))

In [ ]:
messages  = []
messages.append(SystemMessage(content=(
"""
あなたは美容業界とマーケティングに精通したプロのアナリストです。
指定されたコスメブランドについて、客観的かつ詳細な分析レポートを作成してください。

【出力要件】
・最終的に.txtファイルとして出力するため、HTMLタグやMarkdownの特殊な装飾（表組みなど）は使用せず、プレーンテキストとして読みやすいレイアウト（見出し記号や箇条書き）で出力してください。
・論理的で専門的なトーンを保ちつつ、わかりやすい言葉で記述してください。
・冒頭の挨拶や、末尾の「出力が完了しました」などの会話的な文章は一切含めず、レポートの本文のみを出力してください。
"""
			)))
messages.append(UserMessage(content=(
f"""
以下の【対象ブランド】について、【分析項目】に従って詳細な分析レポートを作成してください。

【対象ブランド】
{BRAND_NAME}

【分析項目】
1. ブランド概要
   - ブランドのコンセプト、設立の背景、ビジョン

2. ターゲットペルソナ
   - 主要な顧客層（年齢、性別、価値観、ライフスタイル）
   - 抱えている美容の悩みやニーズ

3. 製品の強みとUSP（独自の売り）
   - 代表的な製品カテゴリや成分の特徴
   - 他社にはない独自の価値（USP）

4. 価格帯とポジショニング
   - 市場における価格帯（プチプラ、ミドプラ、デパコスなど）
   - マトリクス上での立ち位置（例：高価格×自然派 など）

5. マーケティング・販売戦略
   - 主な販売チャネル（オンライン、実店舗、ドラッグストア、百貨店など）
   - プロモーション手法（SNS活用、インフルエンサー、広告戦略など）

6. 競合ブランドとの比較
   - 主要な競合ブランド（2〜3社）とその違い
   - 競合に対する優位性と劣後している点

7. 今後の展望と課題
   - 現在の市場トレンドを踏まえた今後の成長可能性
   - ブランドが直面している、または今後直面する課題

【出力フォーマット】
各項目は見出し（例：■ 1. ブランド概要）を使用し、改行やインデントを適切に用いて見やすく整理してください。
"""
            )))
analysis_data = await llmClient.complete(messages)
analysis_data

In [ ]:
# JSON形式でバックアップ
with open(f'ブランドLP/{BRAND_NAME_FILE}', 'w', encoding='utf-8') as f:
    json.dump(analysis_data, f, indent=4, ensure_ascii=False)
with open(f'ブランドLP/{BRAND_NAME_FILE}', 'r', encoding='utf-8') as f:
    final_data = json.load(f)

final_data